# P2
## Parte 1:

In [3]:
pip install requests

  Using cached requests-2.34.2-py3-none-any.whl.metadata (4.8 kB)
  Using cached charset_normalizer-3.4.7-cp313-cp313-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (40 kB)
  Using cached idna-3.18-py3-none-any.whl.metadata (6.1 kB)
  Using cached urllib3-2.7.0-py3-none-any.whl.metadata (6.9 kB)
Using cached requests-2.34.2-py3-none-any.whl (73 kB)
Using cached charset_normalizer-3.4.7-cp313-cp313-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl (215 kB)
Using cached idna-3.18-py3-none-any.whl (65 kB)
Using cached urllib3-2.7.0-py3-none-any.whl (131 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [requests]

[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [4]:
pip install pymongo

  Using cached dnspython-2.8.0-py3-none-any.whl.metadata (5.7 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 9.7 MB/s  0:00:009.5 MB/s eta 0:00:01
Using cached dnspython-2.8.0-py3-none-any.whl (331 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [pymongo]━━━ 1/2 [pymongo]

[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [8]:
import requests
from pymongo import MongoClient

class MongoDBHandler:
    def __init__(self):
        self.mongo_uri = "mongodb://localhost:27017/"
        self.db_name = "pucp_store"
        self.categories_name = "categorias"
        self.productos_name = "productos"
        self.connect_mongo()

    def connect_mongo(self):
        try:
            client = MongoClient(self.mongo_uri)
            self.db = client[self.db_name] if client else None
            print(f"Conectado a la base de datos '{self.db_name}' en MongoDB.")
        except Exception as e:
            print(f"Error al conectar a MongoDB: {e}")

    def fetch_json_data(self, url):
        try:
            response = requests.get(url)
            response.raise_for_status()
            return response.json()
        except requests.exceptions.HTTPError as errh:
            print("HTTP Error:", errh)
        except requests.exceptions.ConnectionError as errc:
            print("Error de conexión:", errc)
        except requests.exceptions.Timeout as errt:
            print("Timeout Error:", errt)
        except requests.exceptions.RequestException as err:
            print("Error desconocido:", err)
        return None

    def fetch_all_categories(self, base_url):
        data = self.fetch_json_data(base_url)
        if not data:
            print("No se pudo obtener las categorías.")
            return 0
        col = self.db[self.categories_name]
        col.delete_many({})
        col.insert_many(data)
        print(f"Se cargaron {len(data)} categorías en '{self.categories_name}'.")
        return len(data)

    def fetch_all_products(self):
        categorias = list(self.db[self.categories_name].find({}, {"_id": 0}))
        if not categorias:
            print("No hay categorías cargadas. Ejecuta fetch_all_categories primero.")
            return 0
        col = self.db[self.productos_name]
        col.delete_many({})
        total = 0
        for cat in categorias:
            url = f"{cat['url']}?limit=0"
            data = self.fetch_json_data(url)
            if not data or not data.get("products"):
                print(f"  - {cat['slug']}: sin productos")
                continue
            productos = data["products"]
            col.insert_many(productos)
            total += len(productos)
            print(f"  - {cat['slug']}: {len(productos)} productos")
        print(f"Total de productos cargados: {total}")
        return total

In [9]:
mongo_handler = MongoDBHandler()

categories_url = "https://dummyjson.com/products/categories"

mongo_handler.fetch_all_categories(categories_url)
mongo_handler.fetch_all_products()

Conectado a la base de datos 'pucp_store' en MongoDB.
Se cargaron 24 categorías en 'categorias'.
  - beauty: 5 productos
  - fragrances: 5 productos
  - furniture: 5 productos
  - groceries: 27 productos
  - home-decoration: 5 productos
  - kitchen-accessories: 30 productos
  - laptops: 5 productos
  - mens-shirts: 5 productos
  - mens-shoes: 5 productos
  - mens-watches: 6 productos
  - mobile-accessories: 14 productos
  - motorcycle: 5 productos
  - skin-care: 3 productos
  - smartphones: 16 productos
  - sports-accessories: 17 productos
  - sunglasses: 5 productos
  - tablets: 3 productos
  - tops: 5 productos
  - vehicle: 5 productos
  - womens-bags: 5 productos
  - womens-dresses: 5 productos
  - womens-jewellery: 3 productos
  - womens-shoes: 5 productos
  - womens-watches: 5 productos
Total de productos cargados: 194


194

In [10]:
print("Categorías en DB:", mongo_handler.db[mongo_handler.categories_name].count_documents({}))
print("Productos en DB:", mongo_handler.db[mongo_handler.productos_name].count_documents({}))
print("\nEjemplo de categoría:")
print(mongo_handler.db[mongo_handler.categories_name].find_one({}, {"_id": 0}))
print("\nEjemplo de producto (solo algunos campos):")
print(mongo_handler.db[mongo_handler.productos_name].find_one({}, {"_id": 0, "title": 1, "category": 1, "price": 1, "stock": 1}))

Categorías en DB: 24
Productos en DB: 194

Ejemplo de categoría:
{'slug': 'beauty', 'name': 'Beauty', 'url': 'https://dummyjson.com/products/category/beauty'}

Ejemplo de producto (solo algunos campos):
{'title': 'Essence Mascara Lash Princess', 'category': 'beauty', 'price': 9.99, 'stock': 99}
